# Label a text dataset in a grid: SMS spam

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toclocoinc/lattice-grid-notebooks/blob/main/sms-spam-labelling/sms_spam_labelling.ipynb)

Review 150 text messages the way you would in a spreadsheet, but with the rules built in: a **`label` column that is a dropdown of the allowed labels** (`spam`, `ham`, `unsure`), **validation that refuses anything else**, a **needs review** filter for the rows still marked `unsure`, and **keyboard-only labelling** (arrow to a cell, type, Enter, next row). Whatever you enter is written straight back to the DataFrame, and a bar chart of the label counts moves as you type. No API key.

**Data:** [SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), Almeida & Hidalgo, 2011, licensed **CC BY 4.0**; fetched from the UCI archive when you run the notebook (about 200 KB). The notebook code is MIT-licensed (see the repo `LICENSE`).

In [1]:
%pip install -q lattice-grid-jupyter

## 1. Fetch the messages and pre-label the easy ones

We take a seeded sample of 150 messages (50 spam, 100 ham). A crude keyword rule suggests a label for each: two or more spam signals (a prize word, a link, a long number, a pound sign) gives `spam`, none gives `ham`, and a single signal is marked `unsure` **for a person to decide**. The collection's own labels are kept aside in `truth` and not shown in the grid; we only use them at the end to see how the labelling went.

In [2]:
import io, re, urllib.request, zipfile
import pandas as pd

URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
raw = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read())).read("SMSSpamCollection").decode("utf-8")
sms = pd.read_csv(io.StringIO(raw), sep="\t", names=["truth", "text"], quoting=3)
sample = pd.concat([sms[sms.truth == t].sample(n, random_state=7) for t, n in (("spam", 50), ("ham", 100))])
sample = sample.sample(frac=1, random_state=7).reset_index(drop=True)

SIGNALS = r"free|win|won|prize|claim|urgent|txt|stop|www\.|http|\d{5,}|£|\bcall\b"
hits = sample.text.str.lower().str.count(SIGNALS)
sample["label"] = hits.map(lambda n: "spam" if n >= 2 else "ham" if n == 0 else "unsure")
sample.insert(0, "id", sample.index + 1)
print(f"{len(sample)} messages: {sample.label.value_counts().to_dict()}")

150 messages: {'ham': 93, 'spam': 40, 'unsure': 17}


## 2. The labelling grid

`label` is a **lookup column** over the allowed labels, which gives it the dropdown editor and refuses any other value, and a `validation` rule (`oneOf`) backs that up for pasted or filled values. Everything but `label` is read-only. The chart beside it is **bound to the grid**: it counts rows per label in the browser and redraws on every edit.

**Keyboard only:** arrow to a `label` cell, press **Enter** to open the dropdown, type `spam`, `ham` or `unsure`, and press **Enter** again to commit; the cursor drops to the next row, so Enter, type, Enter walks down the column. Paste anything else (say `maybe`, with Ctrl+V or Cmd+V) into a label cell and the cell turns red and keeps its old value. The **Needs review** dropdown shows only the rows still `unsure`, and they leave the list as you label them.

In [3]:
from ipywidgets import Dropdown, HBox, Layout, VBox
from lattice_grid_jupyter import LatticeGridWidget, LatticeChart

LABELS = ["spam", "ham", "unsure"]
grid = LatticeGridWidget(sample[["id", "text", "label"]].assign(messages=1), height=460, columns=[
    {"field": "messages", "layout": {"hidden": True}},  # a 1 per row, for the chart to add up
    {"field": "id", "edit": False, "layout": {"width": 50}},
    {"field": "text", "edit": False, "layout": {"width": 480}},
    {"field": "label", "layout": {"width": 140}, "type": "lookup", "edit": {"enabled": True},
     "lookup": {"options": [{"id": x, "label": x} for x in LABELS]},
     "validation": {"oneOf": LABELS, "message": "Label must be spam, ham or unsure"}},
])
counts = LatticeChart(grid, type="bar", x="label", y="messages", height=460)
grid.layout, counts.layout = Layout(width="70%"), Layout(width="30%")

review = Dropdown(options=["All messages", "Needs review"], description="Show")
def show(change):  # a filter is just grid state
    cond = [{"col": "label", "op": "eq", "value": "unsure"}] if change.new == "Needs review" else []
    grid.state = {**grid.state, "filters": {"op": "and", "conditions": cond}}
review.observe(show, "value")
VBox([review, HBox([grid, counts])])

![the labelling grid](https://raw.githubusercontent.com/toclocoinc/lattice-grid-notebooks/main/sms-spam-labelling/grid.png)

*The grid in JupyterLab on the **Needs review** view after three rows were labelled from the keyboard (14 `unsure` rows left, the chart counting them down). The top cell is red because `maybe` was pasted into it: the validation refused it and the cell kept `unsure`.*

## 3. Get the labelled DataFrame back

`grid.df` is the DataFrame as it stands, with every edit applied (validated values only). It is ordinary pandas from here: how many rows are still `unsure`, and, because we kept the collection's own labels aside, how often your labels agree with them.

In [4]:
labelled = grid.df.drop(columns="messages").merge(sample[["id", "truth"]], on="id")
print(labelled.label.value_counts().to_dict())
done = labelled[labelled.label != "unsure"]
print(f"{len(done)} labelled, {(done.label == done.truth).mean():.1%} agree with the collection's labels")
labelled[labelled.label != labelled.truth].assign(text=lambda d: d.text.str[:70])

{'ham': 93, 'spam': 40, 'unsure': 17}
133 labelled, 98.5% agree with the collection's labels


,id,text,label,truth
2,3,Had your contract mobile 11 Mnths? Latest Moto...,unsure,spam
8,9,Reminder: You have not downloaded the content ...,unsure,spam
13,14,It will stop on itself. I however suggest she ...,unsure,ham
32,33,"New Tones This week include: 1)McFly-All Ab..,...",ham,spam
34,35,R u sure they'll understand that! Wine * good ...,unsure,ham
37,38,The current leading bid is 151. To pause this ...,unsure,spam
42,43,Yup i'm free...,unsure,ham
43,44,Let me know when you've got the money so carlo...,unsure,ham
53,54,(Bank of Granite issues Strong-Buy) EXPLOSIVE ...,ham,spam
63,64,Let's pool our money together and buy a bunch ...,unsure,ham
